In [1]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import os
os.chdir(_root)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110
print("project root:", _root)

project root: /workspaces/geo_distance_arena


# 05 - Scale and Cost

The honest trade-off. Straight-line distance is vectorised and instant; road distance is a shortest-path search per pair and scales far worse. The practitioner's answer is the **hybrid**: a cheap straight-line prefilter to a shortlist, then road distance to rank the shortlist.

In [2]:
import numpy as np
import matplotlib.pyplot as plt

from src.graph.build import load_city_graph
from src.graph.snap import SnapIndex
from src.graph.matrix import nearest_candidates
from src.distance.road import road_distance_m
from src.distance.haversine import haversine_m
from src.evaluation.scale_cost import timing_vs_n
from src.scenarios.points import load_scenario
from src.io_utils import save_fig, save_json
import networkx as nx

drive = load_city_graph('copenhagen', 'drive')
scenario = load_scenario('copenhagen')
customers = [tuple(c) for c in scenario['customers']]

## Compute cost versus N

Straight-line matrix vs road matrix as the point set grows.

In [3]:
timing = timing_vs_n(drive, sizes=[10, 25, 50, 100, 200], seed=42)
save_json(timing.to_dict(orient='records'), '05_timing')
timing

,n,straight_line_s,road_s
0,10,0.000100,0.112568
1,25,0.000149,0.222316
2,50,0.000143,0.387007
3,100,0.000356,0.951372
4,200,0.002100,1.652653


In [4]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(timing['n'], timing['straight_line_s'], marker='o', label='straight-line (vectorised)')
ax.plot(timing['n'], timing['road_s'], marker='o', label='road (Dijkstra per origin)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('N (points per side of the matrix)')
ax.set_ylabel('seconds')
ax.set_title('Straight-line is instant; road distance scales with the network')
ax.legend()
fig.tight_layout()
save_fig(fig, '05_timing')
plt.show()

## The hybrid: straight-line prefilter, road rank

To find each customer's nearest of many candidate stores, a full road search over all stores is wasteful. Prefilter to the k straight-line-nearest with a BallTree, then run road distance only on that shortlist. We measure how often the hybrid agrees with the exhaustive road answer.

In [5]:
import random
rng = random.Random(7)
nodes = list(drive.nodes)
store_nodes = rng.sample(nodes, 40)
store_lats = [drive.nodes[n]['y'] for n in store_nodes]
store_lons = [drive.nodes[n]['x'] for n in store_nodes]
index = SnapIndex.from_graph(drive)
cust_nodes = [index.nearest(lat, lon)[0] for lat, lon in customers]

def road_nearest(cust_node, candidate_nodes):
    best, best_d = None, float('inf')
    for s in candidate_nodes:
        try:
            d = road_distance_m(drive, cust_node, s)
        except nx.NetworkXNoPath:
            continue
        if d < best_d:
            best, best_d = s, d
    return best

k = 5
cust_lats = [lat for lat, lon in customers]
cust_lons = [lon for lat, lon in customers]
idx, _ = nearest_candidates(store_lats, store_lons, cust_lats, cust_lons, k=k)
agree = 0
for i, cust_node in enumerate(cust_nodes):
    full = road_nearest(cust_node, store_nodes)
    shortlist = [store_nodes[j] for j in idx[i]]
    hybrid = road_nearest(cust_node, shortlist)
    agree += int(full == hybrid)
hybrid_stats = {'k': k, 'n_stores': len(store_nodes), 'n_customers': len(cust_nodes),
                'agreement_rate': agree / len(cust_nodes),
                'road_calls_full': len(cust_nodes) * len(store_nodes),
                'road_calls_hybrid': len(cust_nodes) * k}
save_json(hybrid_stats, '05_hybrid')
hybrid_stats

{'k': 5,
 'n_stores': 40,
 'n_customers': 120,
 'agreement_rate': 1.0,
 'road_calls_full': 4800,
 'road_calls_hybrid': 600}

## Rate-limit reality

OSMnx (Overpass) and Nominatim are keyless **fair-use** services, not bulk backends. Download and geocode **once**, cache everything to disk (this project commits the GraphML and the scenario JSON), batch politely with a custom user agent and a delay, and never bulk-geocode the public endpoint. Everything in these notebooks runs offline on the committed caches.

**Takeaway.** Use the vectorised straight-line method (and a BallTree) to go fast and wide; spend road distance only where it changes the decision -- the hybrid gives you both.